# Seance 4 — entrainement du CNN DermaScan (Colab ou poste)

**Regle non negociable : aucun secret Azure dans ce notebook.** Il produit un paquet `paquet_modele_s4.zip` que vous rapatriez ; la publication vers Azure se fait depuis votre poste, deja authentifie.

Sur Colab : Execution > Modifier le type d'execution > GPU (T4). Sans GPU (Colab CPU ou noyau local dans VS Code), le notebook bascule seul en mode degrade : 294 positifs + 5 000 negatifs au lieu de 20 000. Hors Colab, installez d'abord `requirements-entrainement.txt` dans l'environnement du noyau.

In [1]:
import os, sys, torch
try:
    import google.colab
    COLAB = True    # Colab dans le navigateur, ou extension Colab de VS Code
except ImportError:
    COLAB = False   # noyau local (VS Code, Jupyter)
GPU = torch.cuda.is_available()
os.environ['N_NEGATIFS'] = '20000' if GPU else '5000'
print('Colab :', COLAB, '| GPU :', torch.cuda.get_device_name(0) if GPU else 'aucun -> mode degrade CPU',
      '| N_NEGATIFS =', os.environ['N_NEGATIFS'], '| torch', torch.__version__)

Colab : True | GPU : Tesla T4 | N_NEGATIFS = 20000 | torch 2.11.0+cu130


## Points de reprise (obligatoires)

Sur Colab, le point de reprise est ecrit sur votre Drive a chaque epoque. Si la session est coupee, reconnectez-vous et relancez **toutes** les cellules : l'entrainement repart de la derniere epoque terminee. Hors Colab, ou si Drive refuse de se monter, il est ecrit dans `./reprise/`.

In [2]:
DRIVE = False
if COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        DRIVE = True
    except Exception as err:
        print('Drive indisponible :', err)
SAUVEGARDE = '/content/drive/MyDrive/s4-dermascan' if DRIVE else 'reprise'
os.makedirs(SAUVEGARDE, exist_ok=True)
os.environ['REPRISE'] = f'{SAUVEGARDE}/reprise.pt'
print('point de reprise :', os.path.abspath(os.environ['REPRISE']))

Mounted at /content/drive
point de reprise : /content/drive/MyDrive/s4-dermascan/reprise.pt


## Donnees

Vignettes et etiquettes : archive ISIC, publique, CC-BY 4.0, sans compte. Deja presentes dans `donnees/` (copie ou lien), elles ne sont pas retelechargees.

In [3]:
import urllib.request
os.makedirs('donnees', exist_ok=True)
B = 'https://isic-challenge-data.s3.amazonaws.com/2024/'
for f in ['ISIC_2024_Permissive_Training_Input.zip', 'ISIC_2024_Permissive_Training_GroundTruth.csv']:
    if not os.path.exists(f'donnees/{f}'):
        urllib.request.urlretrieve(B + f, f'donnees/{f}.part')   # 623 Mio pour le zip
        os.replace(f'donnees/{f}.part', f'donnees/{f}')         # pas de fichier tronque
print(sorted(os.listdir('donnees')))

['ISIC_2024_Permissive_Training_GroundTruth.csv', 'ISIC_2024_Permissive_Training_Input.zip']


`patient_id` : le `train-metadata.csv` de la competition Kaggle, celui de S2. Le jeton Kaggle n'est **jamais** ecrit en clair dans une cellule :

- Colab navigateur : **Secrets Colab** (icone cle, `KAGGLE_USERNAME` et `KAGGLE_KEY`) ;
- extension Colab de VS Code (Secrets non pris en charge) : saisie masquee ;
- poste : variables d'environnement `KAGGLE_USERNAME` / `KAGGLE_KEY`, ou `~/.kaggle/kaggle.json`.

Sans jeton : copiez le `train-metadata.csv` (ou `.zip`) de votre dossier S2 dans `donnees/`.

In [4]:
meta = [p for p in ('donnees/train-metadata.csv', 'donnees/train-metadata.csv.zip') if os.path.exists(p)]
if not meta and COLAB:
    try:
        from google.colab import userdata
        os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
        os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    except Exception:   # extension VS Code : userdata.get() echoue
        from getpass import getpass
        os.environ['KAGGLE_USERNAME'] = getpass('KAGGLE_USERNAME : ')
        os.environ['KAGGLE_KEY'] = getpass('KAGGLE_KEY : ')
if not meta and (os.environ.get('KAGGLE_KEY') or os.path.exists(os.path.expanduser('~/.kaggle/kaggle.json'))):
    %pip install -q kaggle
    !"{sys.executable}" -m kaggle competitions download -c isic-2024-challenge -f train-metadata.csv -p donnees/
print(sorted(os.listdir('donnees')))

/usr/bin/python3: No module named kaggle.__main__; 'kaggle' is a package and cannot be directly executed
['ISIC_2024_Permissive_Training_GroundTruth.csv', 'ISIC_2024_Permissive_Training_Input.zip']


In [5]:
os.environ['KAGGLE_API_TOKEN'] = os.environ['KAGGLE_KEY']
!kaggle competitions download -c isic-2024-challenge -f train-metadata.csv -p donnees/
!ls -la donnees/

100% 89.2M/89.2M [00:01<00:00, 59.7MB/s]

total 91380
drwxr-xr-x 2 root root     4096 Oct 29 10:25 .
drwxr-xr-x 1 root root     4096 Oct 29 10:23 ..
-rw-r--r-- 1 root root      446 Oct 29 10:23 ISIC_2024_Permissive_Training_GroundTruth.csv
-rw-r--r-- 1 root root      428 Oct 29 10:23 ISIC_2024_Permissive_Training_Input.zip
-rw-r--r-- 1 root root 93555068 Jul 11  2024 train-metadata.csv.zip


In [6]:
!cat donnees/ISIC_2024_Permissive_Training_Input.zip
!echo
!cat donnees/ISIC_2024_Permissive_Training_GroundTruth.csv

<!doctype html>
<meta charset="utf-8">
<meta http-equiv="refresh" content="0; url=https://isic-archive.s3.amazonaws.com/challenges/2024/ISIC_2024_Permissive_Training_Input.zip">
<title>Moved</title>
<p>This file has moved to <a href="https://isic-archive.s3.amazonaws.com/challenges/2024/ISIC_2024_Permissive_Training_Input.zip">https://isic-archive.s3.amazonaws.com/challenges/2024/ISIC_2024_Permissive_Training_Input.zip</a>.

<!doctype html>
<meta charset="utf-8">
<meta http-equiv="refresh" content="0; url=https://isic-archive.s3.amazonaws.com/challenges/2024/ISIC_2024_Permissive_Training_GroundTruth.csv">
<title>Moved</title>
<p>This file has moved to <a href="https://isic-archive.s3.amazonaws.com/challenges/2024/ISIC_2024_Permissive_Training_GroundTruth.csv">https://isic-archive.s3.amazonaws.com/challenges/2024/ISIC_2024_Permissive_Training_GroundTruth.csv</a>.


In [7]:
!rm -f donnees/ISIC_2024_Permissive_Training_*
B2 = 'https://isic-archive.s3.amazonaws.com/challenges/2024/'
for f in ['ISIC_2024_Permissive_Training_Input.zip', 'ISIC_2024_Permissive_Training_GroundTruth.csv']:
    urllib.request.urlretrieve(B2 + f, f'donnees/{f}')
!ls -la donnees/

total 733004
drwxr-xr-x 2 root root      4096 Oct 29 10:27 .
drwxr-xr-x 1 root root      4096 Oct 29 10:23 ..
-rw-r--r-- 1 root root   3914605 Oct 29 10:27 ISIC_2024_Permissive_Training_GroundTruth.csv
-rw-r--r-- 1 root root 653107434 Oct 29 10:27 ISIC_2024_Permissive_Training_Input.zip
-rw-r--r-- 1 root root  93555068 Jul 11  2024 train-metadata.csv.zip


## Code de la seance

Les deux cellules suivantes ecrivent `donnees.py` et `entrainer.py`, identiques a ceux du depot.

In [8]:
%%writefile donnees.py
"""Seance 4 — donnees.

Usage autonome :  python donnees.py
"""
import hashlib
import io
import os
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import StratifiedGroupKFold

DOSSIER = Path(os.environ.get("DONNEES", "donnees"))
ZIP_VIGNETTES = DOSSIER / "ISIC_2024_Permissive_Training_Input.zip"
VERITE = DOSSIER / "ISIC_2024_Permissive_Training_GroundTruth.csv"
N_NEGATIFS = int(os.environ.get("N_NEGATIFS", "5000"))
GRAINE = 42
TAILLE = 128
MOYENNE = [0.485, 0.456, 0.406]
ECART_TYPE = [0.229, 0.224, 0.225]
MANIFESTE = "manifeste_echantillon.csv"


def metadonnees():
    verite = pd.read_csv(VERITE)
    chemin = DOSSIER / "train-metadata.csv"
    if not chemin.exists():
        chemin = DOSSIER / "train-metadata.csv.zip"
    meta = pd.read_csv(chemin, usecols=["isic_id", "patient_id", "target"])
    d = verite.merge(meta, on="isic_id", how="left", validate="one_to_one")
    assert d["patient_id"].notna().all(), "vignettes sans patient_id : mauvais train-metadata.csv ?"
    assert (d["malignant"].astype(int) == d["target"]).all(), "etiquettes discordantes"
    return d.rename(columns={"malignant": "cible"}).astype({"cible": int})[
        ["isic_id", "patient_id", "cible"]]


def echantillonner(d, n_negatifs=N_NEGATIFS, graine=GRAINE):
    pos = d[d.cible == 1]
    neg = d[d.cible == 0].sample(n=n_negatifs, random_state=graine)
    return pd.concat([pos, neg]).sort_values("isic_id").reset_index(drop=True)


def decouper(e, graine=GRAINE):
    plis = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=graine)
    tr, va = next(plis.split(e, e.cible, groups=e.patient_id))
    e = e.copy()
    e["jeu"] = "entrainement"
    e.loc[va, "jeu"] = "validation"
    communs = set(e.patient_id[e.jeu == "entrainement"]) & set(e.patient_id[e.jeu == "validation"])
    assert not communs, f"fuite : {len(communs)} patients des deux cotes"
    return e


def lire_vignette(source, taille=TAILLE):
    im = Image.open(source).convert("RGB").resize((taille, taille), Image.BILINEAR)
    return np.asarray(im, dtype=np.uint8)


def charger_vignettes(ids, taille=TAILLE):
    cache = DOSSIER / f"cache_{len(ids)}_{taille}_{empreinte_ids(ids)[:12]}.npy"
    if cache.exists():
        return np.load(cache)
    x = np.empty((len(ids), taille, taille, 3), dtype=np.uint8)
    with zipfile.ZipFile(ZIP_VIGNETTES) as z:
        for i, isic_id in enumerate(ids):
            x[i] = lire_vignette(io.BytesIO(z.read(f"ISIC_2024_Permissive_Training_Input/{isic_id}.jpg")), taille)
    np.save(cache, x)
    return x


def empreinte_ids(ids):
    return hashlib.sha256("\n".join(ids).encode()).hexdigest()


def vers_tenseur(x_uint8):
    import torch
    t = torch.from_numpy(x_uint8).permute(0, 3, 1, 2).float().div(255)
    m = torch.tensor(MOYENNE).view(1, 3, 1, 1)
    s = torch.tensor(ECART_TYPE).view(1, 3, 1, 1)
    return (t - m) / s


def augmenter(t, generateur):
    import torch
    if torch.rand(1, generator=generateur) < 0.5:
        t = t.flip(3)
    if torch.rand(1, generator=generateur) < 0.5:
        t = t.flip(2)
    return t.rot90(int(torch.randint(0, 4, (1,), generator=generateur)), (2, 3))


def preparer(sortie=Path("model")):
    d = metadonnees()
    e = decouper(echantillonner(d))
    sortie.mkdir(exist_ok=True)
    e.to_csv(sortie / MANIFESTE, index=False)
    return d, e


if __name__ == "__main__":
    d, e = preparer()
    print(f"archive Permissive : {len(d):,} vignettes, {d.cible.sum()} positifs "
          f"({100 * d.cible.mean():.3f} %), {d.patient_id.nunique()} patients")
    print(f"echantillon : {len(e):,} vignettes ({(e.cible == 1).sum()} pos + {(e.cible == 0).sum()} neg, "
          f"graine {GRAINE}), prevalence {100 * e.cible.mean():.2f} %")
    for jeu, g in e.groupby("jeu"):
        print(f"  {jeu:12s} {len(g):5d} vignettes  {g.cible.sum():3d} positifs  "
              f"{g.patient_id.nunique():4d} patients")
    print("patients communs aux deux jeux : 0")
    print(f"manifeste : model/{MANIFESTE}  sha256 "
          f"{hashlib.sha256((Path('model') / MANIFESTE).read_bytes()).hexdigest()[:16]}...")
    x = charger_vignettes(e.isic_id.tolist())
    print(f"vignettes en RAM : {x.shape} {x.dtype}, {x.nbytes / 1e6:.0f} Mo")
    with zipfile.ZipFile(ZIP_VIGNETTES) as z:
        tailles = [Image.open(io.BytesIO(z.read(f"ISIC_2024_Permissive_Training_Input/{i}.jpg"))).size
                   for i in e.isic_id[:500]]
    print(f"taille d'origine (500 premieres) : mediane {np.median([t[0] for t in tailles]):.0f} px, "
          f"min {min(t[0] for t in tailles)}, max {max(t[0] for t in tailles)}")


Writing donnees.py


In [9]:
%%writefile entrainer.py
"""Seance 4 — entrainement ResNet18 sur les vignettes ISIC, puis paquet de modele.

Variables : N_NEGATIFS, EPOQUES_TETE, EPOQUES_DEGEL, REPRISE.
"""
import hashlib
import json
import math
import os
import platform
import resource
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torchvision
from sklearn.metrics import (average_precision_score, confusion_matrix,
                             precision_recall_curve, roc_auc_score)

import donnees as D

VERSION = "2.0.0"
EPOQUES_TETE = int(os.environ.get("EPOQUES_TETE", "4"))
EPOQUES_DEGEL = int(os.environ.get("EPOQUES_DEGEL", "4"))
COUCHES_DEGELEES = ["fc"] + (["layer4"] if EPOQUES_DEGEL else [])
LOT = 64
LR_TETE, LR_DEGEL, WEIGHT_DECAY = 1e-3, 1e-4, 1e-4
RAPPEL_CIBLE = 0.90
REPRISE = Path(os.environ.get("REPRISE", "sorties/reprise.pt"))
MODELE, META = Path("model/modele.pt"), Path("model/modele_meta.json")


def construire(poids_imagenet=True):
    r = torchvision.models.resnet18(weights="IMAGENET1K_V1" if poids_imagenet else None)
    r.fc = torch.nn.Linear(r.fc.in_features, 1)
    return r


def regler_gel(reseau, degel):
    actives = ["fc"] + (["layer4"] if degel else [])
    for nom, p in reseau.named_parameters():
        p.requires_grad = nom.split(".")[0] in actives
    return actives


def mode_entrainement(reseau, actives):
    reseau.eval()
    for nom in actives:
        getattr(reseau, nom).train()


def pauc_80(y, p):
    v = roc_auc_score(1 - y, -p, max_fpr=0.2)
    return 0.5 * 0.2 ** 2 + (0.2 - 0.5 * 0.2 ** 2) / (1.0 - 0.5) * (v - 0.5)


@torch.no_grad()
def logits(reseau, x, dispositif):
    reseau.eval()
    return torch.cat([reseau(D.vers_tenseur(x[i:i + 256]).to(dispositif)).squeeze(1).cpu()
                      for i in range(0, len(x), 256)]).double().numpy()


def seuil_pour_rappel(y, proba, cible):
    precision, rappel, seuils = precision_recall_curve(y, proba)
    i = np.where(rappel >= cible)[0][-1]
    return float(seuils[min(i, len(seuils) - 1)]), float(precision[i]), float(rappel[i])


def sigmoide(z):
    return 1 / (1 + np.exp(-z))


def ram_max_mo():
    r = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    return r / 1e6 if sys.platform == "darwin" else r / 1e3   # octets sur macOS, Ko sur Linux


def main():
    torch.manual_seed(D.GRAINE)
    dispositif = "cuda" if torch.cuda.is_available() else "cpu"
    Path("sorties").mkdir(exist_ok=True)
    t_debut = time.time()

    d, e = D.preparer()
    ids = e.isic_id.tolist()
    x = D.charger_vignettes(ids)
    y = e.cible.to_numpy()
    tr, va = (e.jeu == "entrainement").to_numpy(), (e.jeu == "validation").to_numpy()
    xtr, ytr, xva, yva = x[tr], y[tr], x[va], y[va]
    print(f"[{dispositif}] entrainement {len(xtr)} ({ytr.sum()} pos), validation {len(xva)} "
          f"({yva.sum()} pos), chargement {time.time() - t_debut:.1f} s")

    reseau = construire().to(dispositif)
    perte = torch.nn.BCEWithLogitsLoss()
    gen = torch.Generator().manual_seed(D.GRAINE)
    historique, meilleur, debut, ck = [], {"pr_auc": -1}, 0, None
    if REPRISE.exists():
        ck = torch.load(REPRISE, weights_only=False)
        reseau.load_state_dict(ck["etat"])
        historique, meilleur, debut = ck["historique"], ck["meilleur"], ck["epoque"] + 1
        gen.set_state(ck["generateur"])
        print(f"reprise a l'epoque {debut + 1}")

    optim = None
    for ep in range(debut, EPOQUES_TETE + EPOQUES_DEGEL):
        degel = ep >= EPOQUES_TETE
        actives = regler_gel(reseau, degel)
        if optim is None or ep == EPOQUES_TETE:
            optim = torch.optim.AdamW([p for p in reseau.parameters() if p.requires_grad],
                                      lr=LR_DEGEL if degel else LR_TETE, weight_decay=WEIGHT_DECAY)
            if ck is not None and ep == debut and ck["degel"] == degel:
                optim.load_state_dict(ck["optim"])
        t0 = time.time()
        mode_entrainement(reseau, actives)
        ordre = torch.randperm(len(xtr), generator=gen).numpy()
        cumul = 0.0
        for i in range(0, len(ordre), LOT):
            j = ordre[i:i + LOT]
            xb = D.augmenter(D.vers_tenseur(xtr[j]), gen).to(dispositif)
            yb = torch.from_numpy(ytr[j]).float().to(dispositif)
            optim.zero_grad()
            l = perte(reseau(xb).squeeze(1), yb)
            l.backward()
            optim.step()
            cumul += l.item() * len(j)
        p = sigmoide(logits(reseau, xva, dispositif))
        ligne = {"epoque": ep + 1, "phase": "degel" if degel else "tete",
                 "perte_entrainement": cumul / len(xtr),
                 "pr_auc": average_precision_score(yva, p), "roc_auc": roc_auc_score(yva, p),
                 "pauc_80": pauc_80(yva, p), "duree_s": time.time() - t0}
        historique.append(ligne)
        print(" ".join(f"{k} {v:.4f}" if isinstance(v, float) else f"{k} {v}" for k, v in ligne.items()))
        if ligne["pr_auc"] > meilleur["pr_auc"]:
            meilleur = {"pr_auc": ligne["pr_auc"], "epoque": ep + 1,
                        "etat": {k: v.detach().cpu().clone() for k, v in reseau.state_dict().items()}}
        torch.save({"epoque": ep, "degel": degel, "etat": reseau.state_dict(), "optim": optim.state_dict(),
                    "historique": historique, "meilleur": meilleur, "generateur": gen.get_state()},
                   REPRISE.with_suffix(".tmp"))
        os.replace(REPRISE.with_suffix(".tmp"), REPRISE)

    reseau.load_state_dict(meilleur["etat"])
    REPRISE.unlink(missing_ok=True)
    duree = time.time() - t_debut
    z = logits(reseau, xva, dispositif)
    p = sigmoide(z)

    n_neg_total = int((d.cible == 0).sum())
    beta = D.N_NEGATIFS / n_neg_total
    correction = math.log(beta)
    prev_reelle, prev_ech = float(d.cible.mean()), float(e.cible.mean())
    z_seuil, prec_val, rappel = seuil_pour_rappel(yva, z, RAPPEL_CIBLE)
    seuil_brut, seuil = float(sigmoide(z_seuil)), float(sigmoide(z_seuil + correction))
    p_cal = sigmoide(z + correction)
    decision = z >= z_seuil
    assert ((p_cal >= seuil) == decision).all(), "decisions modifiees"
    vn, fp, fn, vp = confusion_matrix(yva, decision.astype(int), labels=[0, 1]).ravel()
    tfp = fp / (fp + vn)
    prec_reelle = rappel * prev_reelle / (rappel * prev_reelle + tfp * (1 - prev_reelle))

    latence = {}
    for disp in (["cpu", "cuda"] if dispositif == "cuda" else ["cpu"]):
        r = reseau.to(disp).eval()
        xb = D.vers_tenseur(xva[:1]).to(disp)
        with torch.no_grad():
            for _ in range(5):
                r(xb)
            mes = []
            for _ in range(50):
                t0 = time.perf_counter()
                r(xb)
                if disp == "cuda":
                    torch.cuda.synchronize()
                mes.append(1000 * (time.perf_counter() - t0))
        latence[disp] = {"p50_ms": float(np.percentile(mes, 50)), "p95_ms": float(np.percentile(mes, 95))}
    reseau.to("cpu")

    torch.save(reseau.state_dict(), MODELE)
    i_ctrl = int(np.argmax(yva))
    meta = {
        "version": VERSION,
        "architecture": "resnet18",
        "sortie": "1 logit ; probabilite_maligne = sigmoide(logit + correction_logit)",
        "taille_entree": D.TAILLE,
        "normalisation": {"moyenne": D.MOYENNE, "ecart_type": D.ECART_TYPE},
        "redimensionnement": "PIL RGB, resize (taille_entree, taille_entree), BILINEAR",
        "graine": D.GRAINE,
        "echantillonnage": {
            "methode": "tous les positifs + negatifs tires uniformement sans remise (pandas.sample)",
            "source": "ISIC 2024 SLICE-3D Permissive, licence CC-BY 4.0",
            "n_positifs": int((e.cible == 1).sum()), "n_negatifs": D.N_NEGATIFS,
            "n_negatifs_disponibles": n_neg_total, "taux_negatifs": beta, "graine": D.GRAINE,
            "manifeste": D.MANIFESTE,
            "sha256_manifeste": hashlib.sha256((MODELE.parent / D.MANIFESTE).read_bytes()).hexdigest()},
        "decoupage": {"methode": "StratifiedGroupKFold(5) sur patient_id, pli 0 en validation",
                      "n_entrainement": int(tr.sum()), "n_validation": int(va.sum()),
                      "positifs_validation": int(yva.sum()), "patients_communs": 0},
        "entrainement": {"dispositif": dispositif, "epoques_tete": EPOQUES_TETE,
                         "epoques_degel": EPOQUES_DEGEL, "couches_degelees": COUCHES_DEGELEES,
                         "lot": LOT, "lr_tete": LR_TETE, "lr_degel": LR_DEGEL,
                         "weight_decay": WEIGHT_DECAY, "perte": "BCEWithLogitsLoss sans ponderation",
                         "augmentations": ["flip horizontal", "flip vertical", "rotation 90 degres"],
                         "critere_selection": "pr_auc validation", "epoque_retenue": meilleur["epoque"],
                         "duree_totale_s": round(duree, 1),
                         "duree_moyenne_epoque_s": round(float(np.mean([h["duree_s"] for h in historique])), 1),
                         "ram_max_mo": round(ram_max_mo())},
        "prevalence_entrainement": prev_ech,
        "prevalence_reelle": prev_reelle,
        "correction_logit": correction,
        "rappel_cible": RAPPEL_CIBLE,
        "seuil_brut": seuil_brut,
        "seuil": seuil,
        "metriques": {"pr_auc": float(average_precision_score(yva, p)), "roc_auc": float(roc_auc_score(yva, p)),
                      "pauc_80": float(pauc_80(yva, p)), "hasard_pr_auc": float(yva.mean()),
                      "rappel_au_seuil": rappel, "precision_au_seuil_validation": prec_val,
                      "precision_projetee_prevalence_reelle": prec_reelle,
                      "lesions_examinees_par_maligne_reelle": round(1 / prec_reelle),
                      "matrice_confusion_validation": {"VP": int(vp), "FP": int(fp), "FN": int(fn), "VN": int(vn)}},
        "latence_inference_lot1": latence,
        "controle": {"isic_id": e.isic_id[va].iloc[i_ctrl], "probabilite_maligne": round(float(p_cal[i_ctrl]), 6)},
        "sha256_poids": hashlib.sha256(MODELE.read_bytes()).hexdigest(),
        "entraine_le": time.strftime("%Y-%m-%dT%H:%M:%S"),
        "versions": {"python": platform.python_version(), "torch": torch.__version__,
                     "torchvision": torchvision.__version__},
    }
    META.write_text(json.dumps(meta, indent=2, ensure_ascii=False))
    pd.DataFrame(historique).to_csv("sorties/historique.csv", index=False)
    pd.DataFrame({"isic_id": e.isic_id[va].to_numpy(), "cible": yva, "proba_brute": p,
                  "probabilite_maligne": p_cal}).to_csv("sorties/predictions_validation.csv", index=False)
    print(json.dumps({k: meta[k] for k in ("seuil_brut", "seuil", "correction_logit", "metriques",
                                           "latence_inference_lot1", "controle")}, indent=2))
    print(f"duree totale {duree:.0f} s, RAM max {ram_max_mo():.0f} Mo, "
          f"{MODELE} {MODELE.stat().st_size / 1e6:.1f} Mo")


if __name__ == "__main__":
    main()


Writing entrainer.py


## Entrainement

Les lignes d'epoque s'affichent au fil de l'eau. Relevez la duree par epoque et la ligne `latence_inference_lot1` (CPU et GPU) : c'est la comparaison CPU/GPU de la seance.

In [13]:
!"{sys.executable}" -u entrainer.py

[cuda] entrainement 16158 (242 pos), validation 4136 (52 pos), chargement 4.9 s
reprise a l'epoque 2
epoque 2 phase tete perte_entrainement 0.0694 pr_auc 0.1048 roc_auc 0.7965 pauc_80 0.0583 duree_s 22.7909
epoque 3 phase tete perte_entrainement 0.0646 pr_auc 0.1198 roc_auc 0.8043 pauc_80 0.0639 duree_s 22.2476
epoque 4 phase tete perte_entrainement 0.0629 pr_auc 0.1133 roc_auc 0.7911 pauc_80 0.0642 duree_s 16.2304
epoque 5 phase degel perte_entrainement 0.0681 pr_auc 0.1870 roc_auc 0.8509 pauc_80 0.0910 duree_s 18.8835
epoque 6 phase degel perte_entrainement 0.0476 pr_auc 0.1608 roc_auc 0.8644 pauc_80 0.0998 duree_s 21.9107
epoque 7 phase degel perte_entrainement 0.0414 pr_auc 0.1007 roc_auc 0.8248 pauc_80 0.0862 duree_s 21.3147
epoque 8 phase degel perte_entrainement 0.0383 pr_auc 0.2101 roc_auc 0.8779 pauc_80 0.1165 duree_s 21.5056
{
  "seuil_brut": 0.0017155428716489768,
  "seuil": 0.00015822777138716095,
  "correction_logit": -2.38500804256601,
  "metriques": {
    "pr_auc": 0.210

In [12]:
!ls -la /content/drive/MyDrive/s4-dermascan/

total 87491
drwx------ 2 root root     4096 Oct 29 10:33 .
drwx------ 4 root root     4096 Oct 29 10:23 ..
-rw------- 1 root root 89582089 Oct 29 10:33 reprise.pt


## Paquet a rapatrier

Colab navigateur : le zip se telecharge (et une copie reste sur Drive). Extension VS Code ou poste : recuperez-le au chemin affiche (ou sur Drive). Puis sur votre poste : `unzip paquet_modele_s4.zip`, `python verifier_s4.py model`, et comparez `shasum -a 256 model/modele.pt` a `sha256_poids`.

In [14]:
import hashlib, json, shutil, zipfile
with zipfile.ZipFile('paquet_modele_s4.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(os.listdir('model')):
        z.write(f'model/{f}')
    z.write('sorties/historique.csv'); z.write('sorties/predictions_validation.csv')
sha = hashlib.sha256(open('model/modele.pt', 'rb').read()).hexdigest()
assert sha == json.load(open('model/modele_meta.json'))['sha256_poids'], 'poids et meta discordants'
print('sha256_poids', sha, '\npaquet :', os.path.abspath('paquet_modele_s4.zip'))
if DRIVE:
    shutil.copy('paquet_modele_s4.zip', SAUVEGARDE)
if COLAB:
    try:
        from google.colab import files
        files.download('paquet_modele_s4.zip')   # sans effet dans l'extension VS Code
    except Exception as err:
        print('telechargement impossible :', err)

sha256_poids 3f334db3a2a143f1009c3f2e4a88efe76c557d62f7e701a8cb2bb25f3ff501fa 
paquet : /content/paquet_modele_s4.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>